In [14]:

import pandas as pd
import re
import os
import logging
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from py2neo import Graph, Node, Relationship
import time

In [15]:
file_path = "d:\\Users\\LENOVO\\Desktop\\知识图谱\\【脱敏数据】【2017-2024】门诊数据查询.xlsx"

In [16]:
#数据质量评估
# 定义性别特异性诊断
male_only_diag = ['前列腺炎', '阳痿', '早泄', '前列腺增生']
female_only_diag = ['痛经', '月经不调', '盆腔炎', '妊娠剧吐', '乳腺增生']
def assess_raw_data_quality(file_path, processor, output_report=True):
    """
    对原始临床Excel进行质量评估
    """
    # 定义列映射（与 ClinicalDataProcessor 中的一致）
    column_mappings = {
        'tcm_diagnosis': ['中医病名', '中医诊断名称'],
        'tcm_syndrome': ['中医证型', '中医证候', '症候名称'],
        'western_diagnosis': ['诊断', '西医诊断', '西医诊断名称', '临床诊断', '疾病诊断'],
        'symptom': ['主诉', '症状', '临床表现', '主要症状'],
        'tongue_sign': ['舌', '舌象', '舌质', '舌体'],
        'pulse_sign': ['脉', '脉象'],
        'physical_sign': ['体格检查', '体征', '查体', '物理检查'],
        'herb_prescription': ['中药处方明细', '中药处方', '中药'],
        'drug_prescription': ['西药处方明细', '西药处方', '西药'],
        'combined_prescription': ['处方药物', '处方'],
        'chief_complaint': ['主诉', '主要症状'],
        'present_illness': ['现病史', '病史', '发病情况'],
        'past_history': ['既往史', '过往病史', '病史'],
        'coating': ['苔', '舌苔', '苔质']
    }

    # 性别特异性诊断
    male_only_diag = ['前列腺炎', '阳痿', '早泄', '前列腺增生']
    female_only_diag = ['痛经', '月经不调', '盆腔炎', '妊娠剧吐', '乳腺增生']

    all_sheets = pd.read_excel(file_path, sheet_name=None)
    quality_issues = []
    total_rows = 0
    rows_with_issues = 0

    for sheet_name, df in all_sheets.items():
        if df.empty:
            continue

        # 匹配当前工作表中实际存在的列
        matched = {}
        for key, possible_names in column_mappings.items():
            for col in possible_names:
                if col in df.columns:
                    matched[key] = col
                    break

        if not matched:
            continue

        for idx, row in df.iterrows():
            total_rows += 1
            row_issues = {'sheet': sheet_name, 'row': idx + 1}
            has_issue = False

            # 1. 性别-诊断矛盾
            if '性别' in df.columns and 'tcm_diagnosis' in matched:
                gender = row.get('性别')
                diag = row.get(matched['tcm_diagnosis'])
                if pd.notna(gender) and pd.notna(diag):
                    diag_str = str(diag)
                    if gender == '男' and any(kw in diag_str for kw in female_only_diag):
                        row_issues['性别诊断矛盾'] = '男患者出现女性疾病'
                        has_issue = True
                    elif gender == '女' and any(kw in diag_str for kw in male_only_diag):
                        row_issues['性别诊断矛盾'] = '女患者出现男性疾病'
                        has_issue = True

            # 2. 年龄-诊断矛盾（使用医学规则）
            if '年龄' in df.columns and 'western_diagnosis' in matched:
                age = row.get('年龄')
                diag = row.get(matched['western_diagnosis'])
                if pd.notna(age) and pd.notna(diag):
                    try:
                        age_val = float(age)
                        diag_str = str(diag)
                        if age_val < 18 and ('高血压' in diag_str or '冠心病' in diag_str):
                            row_issues['年龄诊断矛盾'] = '未成年患高血压/冠心病'
                            has_issue = True
                        if age_val > 60 and ('水痘' in diag_str or '麻疹' in diag_str):
                            row_issues['年龄诊断矛盾'] = '老年患儿童传染病'
                            has_issue = True
                    except:
                        pass

            # 4. 舌脉-证型矛盾
            if 'tongue_sign' in matched and 'tcm_syndrome' in matched:
                tongue = row.get(matched['tongue_sign'])
                syndrome = row.get(matched['tcm_syndrome'])
                if pd.notna(tongue) and pd.notna(syndrome):
                    tongue_str = str(tongue)
                    syndrome_str = str(syndrome)
                    if '舌红' in tongue_str and '寒' in syndrome_str:
                        row_issues['舌脉证型矛盾'] = '舌红与寒证并存'
                        has_issue = True



            if has_issue:
                rows_with_issues += 1
                quality_issues.append(row_issues)

    report_df = pd.DataFrame(quality_issues)

    if output_report and not report_df.empty:
        report_df.to_csv('quality_issues_report.csv', index=False, encoding='utf-8-sig')
        print(f"质量报告已保存到: quality_issues_report.csv")

    return report_df, total_rows, rows_with_issues

In [17]:
#根据原始数据的形式，添加舌象解析器类
class TongueParser:
    def __init__(self):
        # 舌象属性词库（可拓展）
        self.feature_dict = {
            'color': ['淡红', '淡白', '红', '绛', '青紫', '紫暗', '瘀斑', '瘀点'],
            'shape': ['胖大', '瘦薄', '齿痕', '裂纹', '芒刺', '点刺', '肿胀', '萎缩'],
            'fur_color': ['白', '黄', '灰', '黑', '褐'],
            'fur_quality': ['薄', '厚', '腻', '腐', '燥', '润', '滑', '剥', '无苔', '少苔', '花剥'],
            'other': ['少津', '水滑', '干燥', '颤动', '歪斜', '短缩', '吐弄']
        }
        # 分隔符正则（用于初步分割）
        self.sep_pattern = re.compile(r'[，,;；、\s]+')
        # 各维度关键词按长度降序排序（优先匹配长词）
        self.sorted_features = {
            dim: sorted(words, key=len, reverse=True)
            for dim, words in self.feature_dict.items()
        }

    def parse(self, text: str) -> Dict[str, List[str]]:
        """
        解析舌象描述文本，返回各维度匹配到的关键词列表
        示例：parse("舌淡红，苔薄白，边有齿痕")
        -> {'color': ['淡红'], 'shape': ['齿痕'], 'fur_color': ['白'], 'fur_quality': ['薄'], 'other': []}
        """
        if not text or pd.isna(text):
            return {}
        text = str(text).strip()
        # 统一分隔符为逗号
        for sep in ['；', ';', '、', '，']:
            text = text.replace(sep, ',')
        # 分割片段
        fragments = [frag.strip() for frag in text.split(',') if frag.strip()]
        result = {dim: [] for dim in self.feature_dict}
        for frag in fragments:
            matched = self._match_fragment(frag)
            for dim, words in matched.items():
                result[dim].extend(words)
        # 去重
        for dim in result:
            result[dim] = list(dict.fromkeys(result[dim]))
        return result

    def _match_fragment(self, frag: str) -> Dict[str, List[str]]:
        """在一个片段中匹配各维度关键词（长词优先）"""
        matched = {dim: [] for dim in self.feature_dict}
        remaining = frag
        # 展平所有关键词及其维度
        all_keywords = []
        for dim, words in self.sorted_features.items():
            for w in words:
                all_keywords.append((dim, w))
        all_keywords.sort(key=lambda x: len(x[1]), reverse=True)

        for dim, kw in all_keywords:
            if kw in remaining:
                matched[dim].append(kw)
                remaining = remaining.replace(kw, ' ' * len(kw))  # 占位移除
        return matched
#数据预处理
class ClinicalDataProcessor:
    """通用知识图谱数据清洗与标准化处理器（不含患者节点）"""
    # 创建根日志记录器
    logger = logging.getLogger()
    logger.setLevel(logging.ERROR)  # 只记录ERROR及以上级别

    # 清除所有现有处理器
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)

    # 只添加文件处理器，不添加控制台处理器
    file_handler = logging.FileHandler("clinical_processor_errors.log", encoding='utf-8')
    file_handler.setLevel(logging.ERROR)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    # 禁用第三方库的日志
    logging.getLogger('py2neo').setLevel(logging.ERROR)
    logging.getLogger('urllib3').setLevel(logging.ERROR)
    logging.getLogger('requests').setLevel(logging.ERROR)
    def __init__(self, silent_mode=False):
        self.silent_mode = silent_mode
        self.tongue_parser = TongueParser()
        # 初始化日志记录器
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            ch = logging.StreamHandler()
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            ch.setFormatter(formatter)
            self.logger.addHandler(ch)
             # 中药关键词列表
        self.herb_keywords = {
            "汤", "丸", "散", "膏", "丹", "饮片", "中药", "中成药", "草", "苓", "芪", "芍", "归",
            "地黄", "黄芪", "人参", "甘草", "白术", "茯苓", "当归", "白芍", "川芎", "熟地", "桂枝",
            "麻黄", "柴胡", "半夏", "陈皮", "黄连", "黄芩", "黄柏", "栀子", "大黄", "芒硝", "附子",
            "肉桂", "干姜", "细辛", "吴茱萸", "丁香", "小茴香", "高良姜", "花椒", "胡椒", "荜茇",
            "荜澄茄", "山奈", "艾叶", "炮姜", "灶心土", "大蓟", "小蓟", "地榆", "槐花", "侧柏叶",
            "白茅根", "苎麻根", "羊蹄", "三七", "茜草", "蒲黄", "花蕊石", "降香", "血余炭", "藕节",
            "卷柏", "景天三七", "紫珠", "仙鹤草", "白及", "棕榈炭", "鸡冠花", "花生衣", "断血流",
            "槲寄生", "紫菀", "款冬花", "马兜铃", "枇杷叶", "桑白皮", "葶苈子", "白果", "洋金花",
            "杏仁", "苏子", "百部", "紫苑", "前胡", "白前", "桔梗", "旋覆花", "白芥子", "皂荚",
            "天南星", "白附子", "竹茹", "竹沥", "天竺黄", "海浮石", "海蛤壳", "礞石", "胖大海",
            "昆布", "海藻", "黄药子", "瓦楞子", "蛤壳", "礞石", "胖大海", "木蝴蝶", "冬瓜子",
            "瓜蒌", "川贝母", "浙贝母", "天麻", "钩藤", "石决明", "珍珠母", "牡蛎", "赭石", "蒺藜",
            "罗布麻", "羚羊角", "牛黄", "地龙", "全蝎", "蜈蚣", "僵蚕", "白僵蚕", "蝉蜕", "珍珠",
            "紫贝齿", "代赭石", "磁石", "龙骨", "琥珀", "酸枣仁", "柏子仁", "远志", "合欢皮",
            "夜交藤", "灵芝", "缬草", "首乌藤", "朱砂", "磁石", "龙骨", "琥珀", "酸枣仁", "柏子仁",
            "鹿角霜", "羌活", "独活", "防风", "荆芥", "薄荷", "牛蒡子", "蝉蜕", "桑叶", "菊花",
            "蔓荆子", "淡豆豉", "葛根", "升麻", "枳壳", "防己", "车前子", "金银花", "鹿角",
            "鹿茸", "阿胶", "龟板", "珍珠草", "鳖甲", "海马", "海龙", "蛤蚧", "冬虫夏草", "麦芽"
        }

        self.drug_keywords = {
            "片", "胶囊", "注射液", "颗粒", "软膏", "西药", "化学药", "霉素", "素", "沙星", "唑",
            "酚", "汀", "阿莫西林", "头孢", "青霉素", "布洛芬", "阿司匹林", "维生素", "激素", "抗生素",
            "洛尔", "地平", "普利", "沙坦", "他汀", "西汀", "唑仑", "巴比妥", "卡因", "松", "龙",
            "奈德", "米松", "曲安奈德", "氟轻松", "哈西奈德", "倍他米松", "地塞米松", "泼尼松",
            "氢化可的松", "可的松", "甲基强的松龙", "曲安西龙", "氟米龙", "氯倍他索", "卤米松",
            "氟替卡松", "莫米松", "地奈德", "氟氢可的松", "去炎松", "氟羟氢化泼尼松", "倍氯米松",
            "布地奈德", "环索奈德", "氟尼缩松", "曲安奈德", "氟轻松", "哈西奈德", "阿氯米松",
            "安西奈德", "卤倍他索", "乌倍他索", "氟倍他索", "氯氟舒松", "卤米松", "氟米松",
            "莫米松", "氟替卡松", "地塞米松", "倍他米松", "泼尼松龙", "氢化可的松", "可的松",
            "甲基泼尼松龙", "曲安西龙", "氟氢可的松", "去氧皮质酮", "醛固酮", "氟氢可的松",
            "去氧皮质酮", "醛固酮", "氟氢可的松", "去氧皮质酮", "醛固酮", "贴剂", "喷雾剂", "mg",
            "μg", "ml", "g", "IU", "U", "毫克", "微克", "毫升", "克", "升", "粒", "袋", "包", "支", "瓶",
            "酶", "酮", "苯", "马隆", "葡萄糖", "氨", "酸", "钠", "钾", "钙", "镁", "锌", "铁",
            "西林", "比妥", "妥因", "嘌呤", "嘧啶", "蝶呤", "苷", "肽", "蛋白", "因子", "受体",
            "拮抗剂", "激动剂", "抑制剂", "阻滞剂", "调节剂", "类似物", "衍生物", "前体", "代谢物", "中间体",
            "氢氯噻嗪", "复方丹参"
        }

        # 中药单位
        self.herb_units = {"克", "g", "钱", "两", "分", "斤", "升", "合", "枚", "个", "条", "只", "颗粒"}
        # 西药单位
        self.drug_units = {"mg", "μg", "ml", "IU", "U", "粒", "片", "袋", "包", "支", "瓶", "g", "mg/ml", "μg/ml", "毫克",
                           "微克", "毫升", "国际单位", "单位", "胶囊", "软膏", "贴剂", "喷雾剂"}
        # 实体类型映射
        self.entity_mappings = {
            "tcm_diagnosis": {},  # 中医病名
            "tcm_syndrome": {},  # 中医证型
            "herb": {},  # 中药
            "drug": {},  # 西药
            "tongue_sign": {},  # 舌象
            "pulse_sign": {},  # 脉象
            "physical_sign": {},  # 体格检查
            "western_diagnosis": {},  # 西医诊断
            "symptom": {}  # 症状
        }
        # 实体属性映射
        self.entity_attributes = {
            "tongue_sign": ["coating"],  # 舌象有苔属性
            "pulse_sign": [],
            "tcm_diagnosis": ["present_illness", "past_history"],
            "tcm_syndrome": ["present_illness", "past_history"],
            "symptom": ["chief_complaint"],
            "herb": ["prescription_detail"],
            "drug": ["prescription_detail"]
        }
        # 编译正则表达式
        self.herb_pattern = re.compile(r'([\u4e00-\u9fa5]+)\s*(\d+\.?\d*)\s*[克g钱两分斤升合枚个条只颗粒]')
        self.drug_pattern = re.compile(
            r'([\u4e00-\u9fa5a-zA-Z0-9]+)\s*(\d+\.?\d*)\s*(?:mg|μg|ml|g|IU|U|粒|片|袋|包|支|瓶|毫克|微克|毫升|国际单位|单位)?\s*(?:\d+\s*[片粒支瓶袋]?)?\s*[口服外用每日每次饭前饭后睡前]?')

    # ================== 公共辅助方法 ==================
    def _print_progress(self, message):
        """只在非静默模式下打印进度信息"""
        if not self.silent_mode:
            print(message)

    def _safe_execute(self, func, *args, **kwargs) -> list:
        """安全执行函数，捕获异常，始终返回列表"""
        try:
            result = func(*args, **kwargs)
            if result is None:
                return []
            elif isinstance(result, list):
                return result
            else:
                return [result] if result else []
        except Exception as e:
            error_msg = f"执行 {func.__name__} 时发生错误: {str(e)}"
            self.logger.error(error_msg)
            if not self.silent_mode:
                print(f"警告: {error_msg}")
            return []

    def _split_text(self, text: str) -> List[str]:
        """智能分割临床文本，返回项目列表"""
        if pd.isna(text) or not str(text).strip():
            return []
        text_str = str(text).strip()
        # 尝试多种分隔符
        for sep in ["；", ";", "、", "，", ","]:
            if sep in text_str:
                return [item.strip() for item in text_str.split(sep) if item.strip()]
        # 默认按空白分割
        return [item.strip() for item in re.split(r'\s+', text_str) if item.strip()]

    def _normalize_clinical_text(self, text: str) -> str:
        """标准化临床文本"""
        if pd.isna(text):
            return ""
        text = str(text).lower()
        text = re.sub(r'[^\w\s;；]', '', text)  # 保留分号
        return text.strip()


    def _is_dosage_or_unit(self, text: str) -> bool:
        """检查是否是剂量或单位而非真正的药物名称"""
        if not text or len(text.strip()) == 0:
            return True
        text_str = str(text).strip().lower()
        # 常见剂量/单位模式
        dosage_patterns = [
            r'^\d+\.?\d*$',  # 纯数字
            r'^\d+\.?\d*\s*(?:mg|μg|ml|g|l|ug|mcg|IU|U|毫克|微克|毫升|克|升|%|percent)$',
            r'^\d+\.?\d*\s*(?:片|粒|袋|包|支|瓶|胶囊|颗粒)$',
            r'^\d+\.?\d*\s*/\s*\d+\.?\d*$',  # 比例如 25/75
            r'^\d+%$',
            r'^[一二三四五六七八九十百千万]+$',
            r'^[克gmgμmlIUulL片粒袋包支瓶%]+$',
            r'^q\.?d\.?$|^b\.?i\.?d\.?$|^t\.?i\.?d\.?$|^p\.?r\.?n\.?$',
        ]
        for pattern in dosage_patterns:
            if re.match(pattern, text_str, re.IGNORECASE):
                return True
        # 移除数字和单位后，剩余字符少于2个，也认为是剂量
        cleaned = re.sub(r'[\d\.]+', '', text_str)
        cleaned = re.sub(r'(?:mg|μg|ml|g|l|ug|mcg|IU|U|毫克|微克|毫升|克|升|%|片|粒|袋|包|支|瓶)', '', cleaned,
                         flags=re.IGNORECASE)
        return len(cleaned.strip()) < 2

    def _is_valid_drug_name(self, text: str) -> bool:
        """检查是否是有效的药物名称"""
        if not text or len(str(text).strip()) < 2:
            return False
        text_str = str(text).strip()
        if self._is_dosage_or_unit(text_str):
            return False
        invalid_patterns = [
            r'^\d+$', r'^\d+\.\d+$', r'^\d+\.?\d*(?:mg|μg|ml|g|ug|mcg|IU|U)$',
            r'^[克gmgμmlIUU片粒袋包支瓶%]+$', r'^[一二三四五六七八九十]+$'
        ]
        for pattern in invalid_patterns:
            if re.match(pattern, text_str, re.IGNORECASE):
                return False
        if not re.search(r'[a-zA-Z\u4e00-\u9fa5]', text_str):
            return False
        meaningful_chars = len(re.findall(r'[a-zA-Z\u4e00-\u9fa5]', text_str))
        return meaningful_chars / len(text_str) >= 0.3

    def _create_entity_dict(self, name: str, entity_type: str, row_data: dict,
                            original_text: str = "", extra_attrs: dict = None) -> dict:
        """统一创建实体字典"""
        if name not in self.entity_mappings[entity_type]:
            entity_id = f"{entity_type.upper()}:{abs(hash(name))}"
            self.entity_mappings[entity_type][name] = entity_id
        entity = {
            "entity_id": self.entity_mappings[entity_type][name],
            "name": name,
            "type": entity_type,
            "source": "Clinical Data",
            "source_sheet": row_data.get('source_sheet', ''),
            "source_row": row_data.get('source_row', '')
        }
        if original_text:
            entity["original_text"] = original_text
        if extra_attrs:
            entity.update(extra_attrs)
        # 添加特定属性
        if entity_type in self.entity_attributes:
            for attr in self.entity_attributes[entity_type]:
                if attr in row_data and pd.notna(row_data[attr]):
                    entity[attr] = str(row_data[attr])
        return entity

    def _clean_medication_name(self, name: str, med_type: str) -> str:
        """清洗药物名称（中药或西药）"""
        if not name:
            return ""
        original = str(name)
        # 移除数字剂量
        cleaned = re.sub(r'\d+\.?\d*\s*(?:克|g|钱|两|分|斤|升|合|枚|个|条|只|颗粒|mg|μg|ml|IU|U|毫克|微克|毫升)', '', original)
        # 移除用法说明
        cleaned = re.sub(r'[，,;；、]\s*(?:口服|外用|每日|每次|饭前|饭后|睡前|S\.po|qd|bid|tid).*$', '', cleaned)
        # 移除引号、括号内容（但保留中药炮制标记）
        cleaned = re.sub(r'["\']', '', cleaned)
        # 保留括号内的炮制方法（如“黄芪(炙)”）
        cleaned = re.sub(r'\(([^)]*)\)', r'(\1)', cleaned) if med_type == 'herb' else re.sub(r'[()]', '', cleaned)
        # 移除多余空格和开头结尾非文字字符
        cleaned = re.sub(r'\s+', '', cleaned)
        cleaned = re.sub(r'^[^\w\u4e00-\u9fa5()]+|[^\w\u4e00-\u9fa5()]+$', '', cleaned)
        if not cleaned:
            # 保底：提取连续中文字符或字母数字
            chinese = re.findall(r'[\u4e00-\u9fa5]{2,}', original)
            if chinese:
                cleaned = chinese[0]
            else:
                alnum = re.findall(r'[a-zA-Z0-9]{2,}', original)
                cleaned = alnum[0] if alnum else original[:15]
        return cleaned.strip()

    def _classify_medication_item(self, item_name: str) -> Optional[str]:
        """智能判断药物类型：herb 或 drug"""
        # 优先检查已知中药
        if any(herb in item_name for herb in self.herb_keywords):
            return "herb"
        # 检查已知西药特征
        if any(drug in item_name for drug in self.drug_keywords):
            return "drug"
        # 剂型判断
        chinese_forms = {"汤", "丸", "散", "膏", "丹", "饮片"}
        western_forms = {"片", "胶囊", "注射液", "软膏", "贴剂", "喷雾剂"}
        for form in chinese_forms:
            if item_name.endswith(form):
                return "herb"
        for form in western_forms:
            if item_name.endswith(form):
                return "drug"
        # 颗粒特殊处理
        if "颗粒" in item_name:
            base = item_name.replace("颗粒", "")
            if any(herb in base for herb in self.herb_keywords):
                return "herb"
            if any(drug in item_name for drug in self.drug_keywords):
                return "drug"
            return "herb"  # 默认中药
        return None
    # ================== 实体提取方法 ==================
    def _extract_entity(self, text: str, entity_type: str, row_data: dict) -> list:
        """通用实体提取方法（非药物类）"""
        if pd.isna(text):
            return []
        items = self._split_text(text)
        entities = []
        for item in items:
            if not item:
                continue
            # 移除可能的前置序号
            item = re.sub(r'^\d+[\s、.,，。]*', '', item).strip()
            if not item:
                continue
            normalized = self._normalize_clinical_text(item)
            if normalized:
                entities.append(self._create_entity_dict(normalized, entity_type, row_data, original_text=item))
        return entities

    def _extract_chinese_herbs(self, text: str, row_data: dict) -> list:
        """提取中药实体"""
        if pd.isna(text):
            return []
        text_str = str(text)
        # 预处理
        text_str = re.sub(r'处方\d+[：:]\s*', '', text_str)
        text_str = re.sub(r'\d+剂', '', text_str)
        text_str = re.sub(r'^\d+[\s、.,，。]*', '', text_str)
        text_str = re.sub(r'["\']', '', text_str)
        entities = []
        # 匹配模式：中药名 + 剂量 + 单位
        pattern = re.compile(
            r'([\u4e00-\u9fa5]{2,})'  # 药名（至少2汉字）
            r'\s*(\d+\.?\d*)\s*'  # 剂量
            r'(?:克|g|钱|两|分|斤|升|合|枚|个|条|只|颗粒)'  # 单位
            r'(?:\s|,|;|，|；|、|/|$)'
        )
        for match in pattern.finditer(text_str):
            herb_name = match.group(1).strip()
            dosage = match.group(2).strip()
            cleaned = self._clean_medication_name(herb_name, 'herb')
            if cleaned and len(cleaned) >= 2:
                entities.append(self._create_entity_dict(
                    cleaned, 'herb', row_data,
                    original_text=f"{herb_name} {dosage}克",
                    extra_attrs={'dosage': dosage, 'unit': '克'}
                ))
        # 若无匹配，尝试简单分割
        if not entities:
            items = self._split_text(text_str)
            for item in items:
                if len(item) < 2:
                    continue
                # 判断是否为中药
                if re.search(r'[\u4e00-\u9fa5]{2,}', item) and not self._is_dosage_or_unit(item):
                    cleaned = self._clean_medication_name(item, 'herb')
                    if cleaned and len(cleaned) >= 2:
                        entities.append(self._create_entity_dict(
                            cleaned, 'herb', row_data,
                            original_text=item,
                            extra_attrs={'extraction_method': 'simple'}
                        ))
        return entities

    def _extract_western_drugs(self, text: str, row_data: dict) -> list:
        """提取西药实体"""
        if pd.isna(text):
            return []
        text_str = str(text)
        text_str = re.sub(r'处方\d+[：:]\s*', '', text_str)
        text_str = re.sub(r'^\d+[\s、.,，。]*', '', text_str)
        entities = []
        # 先尝试按逗号分割处理
        items = self._split_text(text_str)
        for item in items:
            if len(item) < 2 or self._is_dosage_or_unit(item):
                continue
            # 匹配模式：药名 + 可能的剂量单位
            drug_pattern = re.compile(
                r'([a-zA-Z\u4e00-\u9fa5]{2,}(?:\([^)]+\))?[a-zA-Z\u4e00-\u9fa5]*(?:片|胶囊|注射液|颗粒|软膏|贴剂|喷雾剂|咀嚼片|乳剂|混悬液|糖浆|口服液|滴剂|栓剂|气雾剂))',
                re.IGNORECASE
            )
            match = drug_pattern.search(item)
            if match:
                drug_name = match.group(1).strip()
                cleaned = self._clean_medication_name(drug_name, 'drug')
                if cleaned and len(cleaned) >= 2 and self._is_valid_drug_name(cleaned):
                    entities.append(self._create_entity_dict(cleaned, 'drug', row_data, original_text=match.group(0)))
            else:
                # 若无匹配，但包含药物特征，直接清洗后加入
                if any(kw in item for kw in self.drug_keywords) and not self._is_dosage_or_unit(item):
                    cleaned = self._clean_medication_name(item, 'drug')
                    if cleaned and len(cleaned) >= 2 and self._is_valid_drug_name(cleaned):
                        entities.append(self._create_entity_dict(
                            cleaned, 'drug', row_data,
                            original_text=item,
                            extra_attrs={'extraction_method': 'fallback'}
                        ))
        return entities

    def _extract_combined_prescription(self, text: str, row_data: dict) -> list:
        """从合并处方中提取药物，并自动分类"""
        if pd.isna(text):
            return []
        text_str = str(text)
        # 预处理
        text_str = re.sub(r'处方\d+[：:]\s*', '', text_str)
        text_str = re.sub(r'^\d+[\s、.,，。]*', '', text_str)
        items = self._split_text(text_str)
        entities = []
        for item in items:
            if len(item) < 2:
                continue
            # 先尝试中药提取
            herb_entities = self._safe_execute(self._extract_chinese_herbs, item, row_data)
            if herb_entities:
                entities.extend(herb_entities)
                continue
            # 再尝试西药提取
            drug_entities = self._safe_execute(self._extract_western_drugs, item, row_data)
            if drug_entities:
                entities.extend(drug_entities)
                continue
            # 若都失败，则智能分类
            med_type = self._classify_medication_item(item)
            if med_type:
                cleaned = self._clean_medication_name(item, med_type)
                if cleaned and len(cleaned) >= 2 and (med_type != 'drug' or self._is_valid_drug_name(cleaned)):
                    entities.append(self._create_entity_dict(
                        cleaned, med_type, row_data,
                        original_text=item,
                        extra_attrs={'extraction_method': 'combined_fallback'}
                    ))
        return entities

    def _extract_prescription_entities(self, text: str, row_data: dict, entity_type: str) -> list:
        """处方实体提取入口，根据类型调用相应方法"""
        if entity_type == "herb":
            return self._extract_chinese_herbs(text, row_data)
        elif entity_type == "drug":
            return self._extract_western_drugs(text, row_data)
        else:
            return self._extract_entity(text, entity_type, row_data)
    def _extract_tongue_entity(self, text: str, row_data: dict) -> list:
        """提取舌象实体，并解析结构化属性存入 extra_attrs"""
        if pd.isna(text):
            return []
        parsed = self.tongue_parser.parse(text)
        extra_attrs = {}
        for dim, values in parsed.items():
            if values:
                extra_attrs[dim] = ';'.join(values)  # 多个值用分号连接
    # 创建舌象实体，名称保留原始描述，属性存入 extra_attrs
        entity = self._create_entity_dict(
            name=text.strip(),
            entity_type='tongue_sign',
            row_data=row_data,
            original_text=text,
            extra_attrs=extra_attrs
        )
        return [entity]
    # ================== 后处理与验证 ==================
    def correct_misclassified_entities(self, entities_df: pd.DataFrame) -> pd.DataFrame:
        """校正明显错误分类的实体（如中药颗粒被误分为西药）"""
        known_chinese_herbs = {name for name in self.entity_mappings['herb'].keys()} | {
            "茯苓颗粒", "吴茱萸颗粒", "黄芪颗粒", "人参颗粒", "甘草颗粒",
            "白术颗粒", "当归颗粒", "白芍颗粒", "川芎颗粒", "熟地颗粒",
            "桂枝颗粒", "麻黄颗粒", "柴胡颗粒", "半夏颗粒", "陈皮颗粒",
            "黄连颗粒", "黄芩颗粒", "黄柏颗粒", "栀子颗粒", "大黄颗粒",
            "羌活颗粒", "独活颗粒", "防风颗粒", "荆芥颗粒", "薄荷颗粒",
            "牛蒡子颗粒", "蝉蜕颗粒", "桑叶颗粒", "菊花颗粒", "蔓荆子颗粒",
            "淡豆豉颗粒", "葛根颗粒", "升麻颗粒", "前胡颗粒", "白前颗粒",
            "杏仁颗粒", "苏子颗粒", "百部颗粒", "紫苑颗粒", "款冬花颗粒",
            "马兜铃颗粒", "枇杷叶颗粒", "桑白皮颗粒", "葶苈子颗粒", "白果颗粒",
            "洋金花颗粒", "旋覆花颗粒", "桔梗颗粒", "竹茹颗粒", "海浮石颗粒",
            "海蛤壳颗粒", "礞石颗粒", "胖大海颗粒", "昆布颗粒", "海藻颗粒",
            "黄药子颗粒", "瓦楞子颗粒", "蛤壳颗粒", "木蝴蝶颗粒", "冬瓜子颗粒",
            "瓜蒌颗粒", "川贝母颗粒", "浙贝母颗粒", "天麻颗粒", "钩藤颗粒",
            "石决明颗粒", "珍珠母颗粒", "牡蛎颗粒", "赭石颗粒", "蒺藜颗粒",
            "罗布麻颗粒", "羚羊角颗粒", "牛黄颗粒", "地龙颗粒", "全蝎颗粒",
            "蜈蚣颗粒", "僵蚕颗粒", "珍珠颗粒", "紫贝齿颗粒", "代赭石颗粒",
            "磁石颗粒", "龙骨颗粒", "琥珀颗粒", "柏子仁颗粒", "合欢皮颗粒",
            "夜交藤颗粒", "灵芝颗粒", "缬草颗粒", "首乌藤颗粒", "朱砂颗粒",
            "酸枣仁颗粒", "枳壳颗粒", "防己颗粒", "车前子颗粒", "金银花颗粒",
            "鹿角颗粒", "鹿角霜颗粒", "鹿茸颗粒", "阿胶颗粒", "龟板颗粒",
            "珍珠草颗粒", "鳖甲颗粒", "海马颗粒", "海龙颗粒", "蛤蚧颗粒",
            "冬虫夏草颗粒", "麦芽颗粒"
        }
        corrected_count = 0
        for idx, row in entities_df.iterrows():
            if row['type'] == 'drug' and row['name'] in known_chinese_herbs:
                entities_df.at[idx, 'type'] = 'herb'
                entities_df.at[idx, 'entity_id'] = f"HERB:{abs(hash(row['name']))}"
                corrected_count += 1
                self.logger.info(f"校正实体分类: {row['name']} 从 drug 改为 herb")
        self.logger.info(f"总共校正了 {corrected_count} 个错误分类的实体")
        return entities_df

    def validate_classification(self, entities_df: pd.DataFrame) -> pd.DataFrame:
        """验证分类结果，自动校正明显错误的中药颗粒"""
        chinese_herbs_in_drug = entities_df[
            (entities_df['type'] == 'drug') &
            (entities_df['name'].str.contains('颗粒')) &
            (~entities_df['name'].str.contains('|'.join(self.drug_keywords)))
        ]
        if not chinese_herbs_in_drug.empty:
            self.logger.warning(f"发现 {len(chinese_herbs_in_drug)} 个可能错误分类的中药颗粒，正在自动校正...")
            for idx, row in chinese_herbs_in_drug.iterrows():
                entities_df.at[idx, 'type'] = 'herb'
                entities_df.at[idx, 'entity_id'] = f"HERB:{abs(hash(row['name']))}"
            self.logger.info(f"自动校正了 {len(chinese_herbs_in_drug)} 个错误分类的中药颗粒")
        return entities_df

    def filter_invalid_drug_entities(self, entities_df: pd.DataFrame) -> pd.DataFrame:
        """过滤无效的药物实体"""
        if entities_df.empty:
            return entities_df
        def is_valid(row):
            name = str(row['name']).strip()
            typ = row['type']
            if typ not in ['drug', 'herb']:
                return True
            if self._is_dosage_or_unit(name):
                return False
            if typ == 'drug' and not self._is_valid_drug_name(name):
                return False
            return len(name) >= 2
        original_count = len(entities_df)
        filtered = entities_df[entities_df.apply(is_valid, axis=1)].copy()
        removed = original_count - len(filtered)
        self.logger.info(f"后处理过滤: 移除 {removed} 个无效实体，剩余 {len(filtered)} 个")
        return filtered
    # ================== 核心处理流程 ==================
    def process_clinical_excel(self, file_path: str) -> pd.DataFrame:
        """处理临床Excel文件，返回实体DataFrame"""
        try:
            all_sheets = pd.read_excel(file_path, sheet_name=None)
            self.logger.info(f"成功读取临床Excel文件: {file_path}，包含 {len(all_sheets)} 个工作表")
        except Exception as e:
            self.logger.error(f"读取文件失败: {str(e)}")
            return pd.DataFrame()

        # 列名映射（支持多种变体）
        column_mappings = {
            'tcm_diagnosis': ['中医病名', '中医诊断名称'],
            'tcm_syndrome': ['中医证型', '中医证候', '症候名称'],
            'western_diagnosis': ['诊断', '西医诊断', '西医诊断名称', '临床诊断', '疾病诊断'],
            'symptom': ['主诉', '症状', '临床表现', '主要症状'],
            'tongue_sign': ['舌', '舌象', '舌质', '舌体'],
            'pulse_sign': ['脉', '脉象'],
            'physical_sign': ['体格检查', '体征', '查体', '物理检查'],
            'herb_prescription': ['中药处方明细', '中药处方', '中药'],
            'drug_prescription': ['西药处方明细', '西药处方', '西药'],
            'combined_prescription': ['处方药物', '处方'],
            'chief_complaint': ['主诉', '主要症状'],
            'present_illness': ['现病史', '病史', '发病情况'],
            'past_history': ['既往史', '过往病史', '病史'],
            'coating': ['苔', '舌苔', '苔质']
        }

        all_entities = []
        entity_locations = defaultdict(list)  # 记录每个实体出现的 (sheet, row)

        for sheet_name, df in all_sheets.items():
            if df.empty:
                self.logger.warning(f"工作表 '{sheet_name}' 为空，跳过")
                continue

            # 匹配实际存在的列
            matched = {}
            for key, possible_names in column_mappings.items():
                for col in possible_names:
                    if col in df.columns:
                        matched[key] = col
                        break
            if not matched:
                self.logger.warning(f"工作表 '{sheet_name}' 无匹配列，跳过")
                continue

            self.logger.info(f"处理工作表: {sheet_name}，匹配列: {matched}")
            total_rows = len(df)
            for idx, row in df.iterrows():
                row_data = {'source_sheet': sheet_name, 'source_row': idx + 1}
                # 添加所有匹配列的值
                for key, col in matched.items():
                    if pd.notna(row.get(col)):
                        row_data[key] = str(row[col])

                entities = []
                # 处理非处方实体（直接分割）
                for ent_type in ['tcm_diagnosis', 'tcm_syndrome', 'western_diagnosis',
                 'symptom', 'pulse_sign', 'physical_sign']:
                    if ent_type in matched and ent_type in row_data:
                        text = row_data[ent_type]
                        if text and str(text).strip():
                            entities.extend(self._safe_execute(self._extract_entity, text, ent_type, row_data))

                # 单独处理舌象（使用专用解析器）
                if 'tongue_sign' in matched and 'tongue_sign' in row_data:
                    text = row_data['tongue_sign']
                    if text and str(text).strip():
                        entities.extend(self._safe_execute(self._extract_tongue_entity, text, row_data))

                # 处理处方实体
                if 'herb_prescription' in matched and 'herb_prescription' in row_data:
                    text = row_data['herb_prescription']
                    if text and str(text).strip():
                        entities.extend(self._safe_execute(self._extract_prescription_entities, text, row_data, "herb"))

                if 'drug_prescription' in matched and 'drug_prescription' in row_data:
                    text = row_data['drug_prescription']
                    if text and str(text).strip():
                        entities.extend(self._safe_execute(self._extract_prescription_entities, text, row_data, "drug"))

                if 'combined_prescription' in matched and 'combined_prescription' in row_data:
                    text = row_data['combined_prescription']
                    if text and str(text).strip():
                        entities.extend(self._safe_execute(self._extract_combined_prescription, text, row_data))

                # 记录实体位置
                for ent in entities:
                    if 'source_sheet' in ent and 'source_row' in ent:
                        entity_locations[ent['entity_id']].append((ent['source_sheet'], ent['source_row']))

                all_entities.extend(entities)
                if (idx + 1) % 100 == 0:
                    self._print_progress(f"工作表 '{sheet_name}': 已处理 {idx+1}/{total_rows} 行")

        if not all_entities:
            return pd.DataFrame()

        # 构建DataFrame并去重合并属性
        entities_df = pd.DataFrame(all_entities)
        # 按entity_id合并属性
        grouped = entities_df.groupby('entity_id')
        merged = []
        for eid, group in grouped:
            base = group.iloc[0].to_dict()
            # 合并多值属性（用分号连接）
            for col in group.columns:
                if col in ['entity_id', 'name', 'type', 'source']:
                    continue
                values = set()
                for val in group[col].dropna():
                    if ';' in str(val):
                        values.update(str(val).split(';'))
                    else:
                        values.add(str(val))
                if values:
                    base[col] = ';'.join(sorted(values))
            merged.append(base)
        entities_df = pd.DataFrame(merged)
        # 后处理
        entities_df = self.validate_classification(entities_df)
        entities_df = self.filter_invalid_drug_entities(entities_df)

        # 将实体位置信息存入实例变量，供后续关系创建使用
        self.entity_locations = entity_locations

        self._print_progress(f"实体提取完成：共 {len(entities_df)} 个唯一实体")
        return entities_df

    # ================== 关系创建 ==================
    def create_clinical_edges(self, clinical_entities: pd.DataFrame,
                              entity_locations: dict = None) -> pd.DataFrame:
        """创建实体间的关系边（基于共现和重要性）"""
        if clinical_entities.empty:
            return pd.DataFrame()
        if entity_locations is None:
            entity_locations = getattr(self, 'entity_locations', {})

        # 按类型组织实体
        entities_by_type = defaultdict(list)
        entity_importance = clinical_entities['entity_id'].value_counts().to_dict()
        for _, row in clinical_entities.iterrows():
            entities_by_type[row['type']].append(row)

        # 关系类型及每个源实体的目标数量上限
        sampling_limits = {
            ('tcm_diagnosis', 'symptom'): 10,
            ('tcm_diagnosis', 'tongue_sign'): 5,
            ('tcm_diagnosis', 'pulse_sign'): 5,
            ('tcm_diagnosis', 'herb'): 15,
            ('tcm_diagnosis', 'drug'): 8,
            ('symptom', 'herb'): 5,
            ('tcm_diagnosis', 'western_diagnosis'): 5,
            ('pulse_sign', 'symptom'): 20,
            ('tongue_sign', 'symptom'): 20,
        }

        edges = []
        # 先处理基于重要性的关系
        for (src_type, tgt_type), limit in sampling_limits.items():
            src_entities = entities_by_type.get(src_type, [])
            tgt_entities = entities_by_type.get(tgt_type, [])
            if not src_entities or not tgt_entities:
                continue
            for src in src_entities:
                src_id = src['entity_id']
                # 按重要性排序目标实体
                sorted_tgts = sorted(tgt_entities,
                                     key=lambda x: entity_importance.get(x['entity_id'], 1),
                                     reverse=True)
                for tgt in sorted_tgts[:limit]:
                    tgt_id = tgt['entity_id']
                    relation = self._get_relation_type(src_type, tgt_type)
                    edges.append({
                        'source': src_id,
                        'target': tgt_id,
                        'relation': relation,
                        'source_type': src_type,
                        'target_type': tgt_type,
                        'source_dataset': 'Clinical Data',
                        'weight': entity_importance.get(src_id, 1) * entity_importance.get(tgt_id, 1)
                    })

        # 特殊处理：中医诊断与西医诊断的共现关系
        tcm_entities = entities_by_type.get('tcm_diagnosis', [])
        west_entities = entities_by_type.get('western_diagnosis', [])
        if tcm_entities and west_entities and entity_locations:
            cooccur = defaultdict(int)
            for tcm in tcm_entities:
                tcm_id = tcm['entity_id']
                tcm_locs = set(entity_locations.get(tcm_id, []))
                for west in west_entities:
                    west_id = west['entity_id']
                    west_locs = set(entity_locations.get(west_id, []))
                    common = tcm_locs & west_locs
                    if common:
                        cooccur[(tcm_id, west_id)] += len(common)
            for (tcm_id, west_id), cnt in cooccur.items():
                if cnt > 1:  # 至少共现2次才建立关系
                    edges.append({
                        'source': tcm_id,
                        'target': west_id,
                        'relation': 'corresponds_to',
                        'source_type': 'tcm_diagnosis',
                        'target_type': 'western_diagnosis',
                        'source_dataset': 'Clinical Data',
                        'weight': cnt,
                        'evidence': f'co-occurrence_in_{cnt}_records'
                    })

        edges_df = pd.DataFrame(edges) if edges else pd.DataFrame()
        # 去重（同一对实体可能有多条同类型关系，保留一条即可）
        if not edges_df.empty:
            edges_df = edges_df.drop_duplicates(subset=['source', 'relation', 'target'])
        self._print_progress(f"关系创建完成：共 {len(edges_df)} 条边")
        return edges_df

    def _get_relation_type(self, source_type: str, target_type: str) -> str:
        """根据实体类型确定关系类型"""
        relation_map = {
            ('tcm_diagnosis', 'symptom'): 'has_symptom',
            ('tcm_diagnosis', 'tongue_sign'): 'has_sign',
            ('tcm_diagnosis', 'pulse_sign'): 'has_sign',
            ('tcm_diagnosis', 'herb'): 'treated_with_herb',
            ('tcm_diagnosis', 'drug'): 'treated_with_drug',
            ('symptom', 'herb'): 'treated_by_herb',
            ('tcm_diagnosis', 'western_diagnosis'): 'corresponds_to',
            ('pulse_sign', 'symptom'): 'indicates_symptom',
            ('tongue_sign', 'symptom'): 'indicates_symptom',
            ('symptom', 'pulse_sign'): 'associated_with_pulse',
            ('symptom', 'tongue_sign'): 'associated_with_tongue',
        }
        return relation_map.get((source_type, target_type), 'related_to')

    # ================== 输出与保存 ==================
    def generate_entity_table(self, clinical_entities: pd.DataFrame) -> pd.DataFrame:
        """生成包含所有实体及其属性的完整表格"""
        if clinical_entities.empty:
            return pd.DataFrame()
        core_cols = ['entity_id', 'name', 'type', 'source']
        other_cols = [c for c in clinical_entities.columns if c not in core_cols]
        ordered = core_cols + sorted(other_cols)
        table = clinical_entities[ordered].copy()
        table['description'] = table['type'] + ': ' + table['name']
        return table.fillna('')

    def generate_kg_files(self, clinical_entities: pd.DataFrame,
                          clinical_edges: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """生成知识图谱节点和边文件"""
        # 节点文件
        keep_cols = ['entity_id', 'name', 'type', 'source',
                     'chief_complaint', 'coating', 'present_illness',
                     'past_history', 'prescription_detail',
                     'source_sheet', 'source_row']
        avail_cols = [c for c in keep_cols if c in clinical_entities.columns]
        nodes = clinical_entities[avail_cols].copy().rename(columns={
            'entity_id': 'node_id', 'name': 'node_name',
            'type': 'node_type', 'source': 'node_source'
        })
        nodes['node_description'] = nodes['node_type'] + ': ' + nodes['node_name']

        # 边文件
        if not clinical_edges.empty:
            edges = clinical_edges[['source', 'target', 'relation', 'source_type', 'target_type']].copy().rename(
                columns={'source': 'source_id', 'target': 'target_id', 'relation': 'relation_type'})
            edges['edge_source'] = 'Clinical Data'
            edges['evidence'] = 'clinical_record'
        else:
            edges = pd.DataFrame()
        return nodes, edges

    def save_entity_table(self, entity_table: pd.DataFrame, output_dir: str,
                          file_name: str = "clinical_entities_full.csv") -> Optional[str]:
        """保存实体表格到CSV文件"""
        if entity_table.empty:
            self._print_progress("没有实体数据可保存")
            return None
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        path = os.path.join(output_dir, file_name)
        entity_table.to_csv(path, index=False, encoding='utf-8-sig')
        self._print_progress(f"实体表格已保存到: {path}")
        return path


In [18]:
# ================== 导入Neo4j ==================
def create_robust_connection(max_retries=5, retry_delay=3, silent_mode=True):
    """创建健壮的Neo4j连接"""
    for attempt in range(max_retries):
        try:
            graph = Graph(
                "bolt://localhost:7687",
                auth=("neo4j", "050210Swh"),
            )
            # 简单的连接测试
            graph.run("RETURN 1").data()
            print("✓ 成功连接到Neo4j")
            return graph
        except Exception as e:
            print(f"连接尝试 {attempt + 1}/{max_retries} 失败: {str(e)}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise Exception(f"无法连接到Neo4j数据库: {str(e)}")

def clear_database_safe(graph):
    """安全清空数据库"""
    try:
        print("正在清空数据库...")
        # 分批删除，避免内存问题
        while True:
            result = graph.run("MATCH (n) WITH n LIMIT 1000 DETACH DELETE n RETURN count(n) as deleted").data()
            deleted = result[0]['deleted'] if result else 0
            if deleted == 0:
                break
            print(f"已删除 {deleted} 个节点...")
        print("✓ 数据库已清空")
    except Exception as e:
        raise Exception(f"清空数据库失败: {str(e)}")


def import_nodes_safe(graph, nodes_df, silent_mode=True):
    """安全导入节点"""
    if not silent_mode:
        print(f"开始导入 {len(nodes_df)} 个节点...")

    try:
        graph.run("CREATE INDEX IF NOT EXISTS FOR (n) ON (n.node_id)")
    except:
        pass  # 索引创建失败不影响后续操作

    batch_size = 100
    total_nodes = len(nodes_df)
    imported = 0
    errors = 0

    for start in range(0, total_nodes, batch_size):
        end = min(start + batch_size, total_nodes)
        batch = nodes_df.iloc[start:end]

        try:
            tx = graph.begin()
            for _, row in batch.iterrows():
                try:
                    labels = (row['node_type'],)
                    properties = {
                        'node_id': row['node_id'],
                        'name': row['node_name'],
                        'source': row.get('node_source', ''),
                        'description': row.get('node_description', '')
                    }

                    for col in batch.columns:
                        if col not in ['node_id', 'node_name', 'node_type', 'node_source', 'node_description']:
                            if pd.notna(row[col]):
                                properties[col] = row[col]

                    node = Node(*labels, **properties)
                    tx.create(node)
                    imported += 1
                except Exception as e:
                    errors += 1

            tx.commit()
            if not silent_mode:
                print(f"已导入 {end}/{total_nodes} 个节点")
        except Exception as e:
            if 'tx' in locals():
                tx.rollback()
            errors += batch_size

    if not silent_mode:
        print(f"节点导入完成: 成功 {imported}, 失败 {errors}")
    return imported, errors


def import_relationships_safe(graph, edges_df, silent_mode=True):
    """安全导入关系"""
    if edges_df.empty:
        if not silent_mode:
            print("没有关系数据可导入")
        return 0, 0

    if not silent_mode:
        print(f"开始导入 {len(edges_df)} 个关系...")

    batch_size = 200
    total_edges = len(edges_df)
    imported = 0
    errors = 0

    for start in range(0, total_edges, batch_size):
        end = min(start + batch_size, total_edges)
        batch = edges_df.iloc[start:end]

        try:
            tx = graph.begin()
            for _, row in batch.iterrows():
                try:
                    source_node = graph.nodes.match(node_id=row['source_id']).first()
                    target_node = graph.nodes.match(node_id=row['target_id']).first()

                    if not source_node or not target_node:
                        errors += 1
                        continue

                    rel_type = row['relation_type']
                    rel = Relationship(source_node, rel_type, target_node)

                    rel_properties = {
                        'source_type': row.get('source_type', ''),
                        'target_type': row.get('target_type', ''),
                        'edge_source': row.get('edge_source', 'Clinical Data'),
                        'evidence': row.get('evidence', 'clinical_record')
                    }
                    rel.update(rel_properties)

                    tx.create(rel)
                    imported += 1
                except Exception as e:
                    errors += 1

            tx.commit()
            if not silent_mode:
                print(f"已导入 {end}/{total_edges} 个关系")
        except Exception as e:
            if 'tx' in locals():
                tx.rollback()
            errors += batch_size

    if not silent_mode:
        print(f"关系导入完成: 成功 {imported}, 失败 {errors}")
    return imported, errors
# 检查实体重复
def deduplicate_entities(nodes_df, edges_df, similarity_threshold=0.8):
    """合并高度相似的实体"""
    # 找到高度相似的实体对
    duplicate_pairs = []

    # 创建实体名称到ID的映射
    name_to_id = {}
    for _, row in nodes_df.iterrows():
        name_to_id[row['node_name']] = row['node_id']

    # 简单的基于名称的重复检测（实际应该用更复杂的方法）
    base_names = {}
    for name in nodes_df['node_name']:
        # 移除括号内容获取基础名称
        base_name = re.sub(r'\([^)]*\)', '', name).strip()
        if base_name not in base_names:
            base_names[base_name] = []
        base_names[base_name].append(name)

    # 合并重复实体
    merged_entities = []
    merged_edges = edges_df.copy()

    for base_name, variants in base_names.items():
        if len(variants) > 1:
            print(f"发现重复实体: {base_name} -> {variants}")
            # 选择第一个作为代表
            main_entity = variants[0]
            main_id = name_to_id[main_entity]

            # 合并其他变体
            for variant in variants[1:]:
                variant_id = name_to_id[variant]
                # 在边表中更新引用
                merged_edges['source_id'] = merged_edges['source_id'].replace(variant_id, main_id)
                merged_edges['target_id'] = merged_edges['target_id'].replace(variant_id, main_id)

    # 移除重复的边
    merged_edges = merged_edges.drop_duplicates()

    # 更新节点表
    unique_entities = set()
    final_nodes = []
    for _, row in nodes_df.iterrows():
        base_name = re.sub(r'\([^)]*\)', '', row['node_name']).strip()
        if base_name not in unique_entities:
            unique_entities.add(base_name)
            final_nodes.append(row)

    return pd.DataFrame(final_nodes), merged_edges

In [ ]:
# print("ClinicalDataProcessor 的方法列表：", [m for m in dir(ClinicalDataProcessor) if not m.startswith('_')])
# import inspect
# print(inspect.getsource(ClinicalDataProcessor))
# ================== 主程序部分 ==================
def main():
    """主程序 - 改进版本"""
    sample = pd.read_excel(file_path, sheet_name=None)
    for sh, df in sample.items():
        print(f"\n=== 工作表 {sh} ===")
        for col in ['中药处方明细', '西药处方明细', '处方药物']:
            if col in df.columns:
                print(f"{col} 样例：", df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else "全空")
        break
    try:
        print("========== 临床数据知识图谱处理程序 ==========")
        print("启动程序...")

        # 初始化处理器 - 默认静默模式
        processor = ClinicalDataProcessor(silent_mode=False)
        # 定义质量阈值（可根据需要调整）
        QUALITY_THRESHOLD = 0.2  # 问题记录占比超过20%则警告

        # 执行质量评估
        print("正在进行数据质量评估...")
        report, total_rows, issue_rows = assess_raw_data_quality(
            file_path,  # 您的Excel文件路径
            processor,
            output_report=True
        )

        # 打印评估摘要
        issue_rate = issue_rows / total_rows if total_rows > 0 else 0
        print("\n========== 数据质量评估结果 ==========")
        print(f"总记录数: {total_rows}")
        print(f"发现问题记录数: {issue_rows} (占比 {issue_rate:.2%})")

        if not report.empty:
            # 统计各类问题数量
            issue_cols = [c for c in report.columns if c not in ['sheet', 'row']]
            print("\n各类问题统计:")
            for col in issue_cols:
                count = report[col].notna().sum()
                print(f"  {col}: {count} 条")
        else:
            print("未发现任何质量问题！")

        # 判断数据是否适合构建知识图谱
        if issue_rate == 0:
            quality_verdict = "✅ 数据质量优秀，完全适合构建知识图谱"
        elif issue_rate < 0.1:
            quality_verdict = "✅ 数据质量良好，适合构建知识图谱（问题记录<10%）"
        elif issue_rate < 0.2:
            quality_verdict = "⚠️ 数据质量一般，建议清洗后构建图谱（问题记录10%~20%）"
        elif issue_rate < 0.3:
            quality_verdict = "⚠️ 数据质量问题较多，需要先进行数据清洗（问题记录20%~30%）"
        else:
            quality_verdict = "❌ 数据质量问题严重，不建议直接构建图谱，请检查数据来源（问题记录>30%）"

        print(f"\n【结论】{quality_verdict}")


        # 处理临床数据
        print("正在处理临床数据...")
        clinical_entities = processor.process_clinical_excel(file_path)
        # 校正明显错误分类的实体
        clinical_entities = processor.correct_misclassified_entities(clinical_entities)
            # 验证分类结果
        clinical_entities = processor.validate_classification(clinical_entities)
        # 按工作表统计
        if 'source_sheet' in clinical_entities.columns:
            sheet_counts = clinical_entities['source_sheet'].value_counts()
            print(f"工作表分布: {dict(sheet_counts)}")
        #显示实体类型统计
        print("=== 实体类型统计 ===")
        if not clinical_entities.empty:
            type_counts = clinical_entities['type'].value_counts()
            print(type_counts)

            # 专门检查drug实体
            drug_entities = clinical_entities[clinical_entities['type'] == 'drug']
            print(f"\nDrug实体数量: {len(drug_entities)}")
            if len(drug_entities) > 0:
                print("Drug实体样例:")
                print(drug_entities[['entity_id', 'name', 'original_text']].head())
            else:
                print("警告: 没有提取到drug实体!")

        print(f"✓ 成功提取 {len(clinical_entities)} 个实体")

        # 创建关系边
        print("正在创建关系边...")
        start_time = time.time()
        print("使用改进重要性算法创建关系...")
        edges = processor.create_clinical_edges(clinical_entities)
        end_time = time.time()
        print(f"✓ 关系创建完成，耗时 {(end_time - start_time):.2f} 秒")

        if not edges.empty:
            print(f"✓ 成功创建 {len(edges)} 个关系边")
            # 显示关系类型统计
            relation_counts = edges['relation'].value_counts()
            print("\n=== 关系类型统计 ===")
            print(relation_counts)
        else:
            print("警告: 未创建任何关系边")

        print("\n" + "=" * 50)
        print("开始实体去重处理")
        print("=" * 50)

        # 生成临时的知识图谱文件用于去重
        temp_kg_nodes, temp_kg_edges = processor.generate_kg_files(clinical_entities, edges)

        print(f"去重前统计:")
        print(f"- 节点数: {len(temp_kg_nodes)}")
        print(f"- 边数: {len(temp_kg_edges)}")

        # 执行实体去重
        kg_nodes_deduplicated, kg_edges_deduplicated = deduplicate_entities(temp_kg_nodes, temp_kg_edges)

        print(f"去重后统计:")
        print(f"- 节点数: {len(kg_nodes_deduplicated)}")
        print(f"- 边数: {len(kg_edges_deduplicated)}")
        print(f"- 移除重复节点: {len(temp_kg_nodes) - len(kg_nodes_deduplicated)}")
        print(f"- 移除重复边: {len(temp_kg_edges) - len(kg_edges_deduplicated)}")

        # 使用去重后的数据继续后续处理
        kg_nodes = kg_nodes_deduplicated
        kg_edges = kg_edges_deduplicated

        # 保存文件
        output_path = r'd:\Users\LENOVO\Desktop\知识图谱'
        entity_table_path = processor.save_entity_table(clinical_entities, output_path, "clinical_entities_full.csv")
        if entity_table_path:
            print(f"✓ 实体表格已保存: {entity_table_path}")

        nodes_path = os.path.join(output_path, "clinical_kg_nodes.csv")
        edges_path = os.path.join(output_path, "clinical_kg_edges.csv")

        Path(output_path).mkdir(parents=True, exist_ok=True)

        # 保存节点文件
        kg_nodes.to_csv(nodes_path, index=False, encoding='utf-8-sig')
        print(f"✓ 节点文件已保存: {nodes_path}")

        # 保存边文件
        if not kg_edges.empty:
            kg_edges.to_csv(edges_path, index=False, encoding='utf-8-sig')
            print(f"✓ 边文件已保存: {edges_path}")
        else:
            print("⚠ 警告: 边数据为空，未保存边文件")
        # 导入到Neo4j
        print("\n是否导入到Neo4j数据库? (y/n): ", end="")
        choice = input().lower()

        if choice == 'y':
            print("正在连接Neo4j数据库...")
            try:
                graph = create_robust_connection(silent_mode=True)
                print("✓ 成功连接到Neo4j")
                print("正在清空数据库...")
                # 清空数据库
                clear_database_safe(graph)
                print("✓ 数据库已清空")

                print("正在导入节点...")
                node_imported, node_errors = import_nodes_safe(graph, kg_nodes, silent_mode=True)
                print(f"✓ 节点导入完成: 成功 {node_imported}, 失败 {node_errors}")

                print("正在导入关系...")
                edge_imported, edge_errors = import_relationships_safe(graph, kg_edges, silent_mode=True)
                print(f"✓ 关系导入完成: 成功 {edge_imported}, 失败 {edge_errors}")

            except Exception as e:
                print(f"Neo4j导入失败: {str(e)}")
                print("CSV文件已保存，可以稍后手动导入")

        print("\n========== 处理完成! ==========")
        print("程序执行成功，知识图谱已生成")
        input("按回车键退出...")

    except KeyboardInterrupt:
        print("\n程序被用户中断")
        input("按回车键退出...")
if __name__ == "__main__":
    main()


=== 工作表 2017 ===
处方药物 样例： 处方1:二甲双胍肠溶片 250mg 42片 S.po tid 500mg 精蛋白锌(25R)赖脯胰岛素混合注射液(优伴) 3ml:300iu 1瓶 S.H bid 24iu BD胰岛素注射笔针头 1个 14个 S.H bid 1个
========== 临床数据知识图谱处理程序 ==========
启动程序...
正在进行数据质量评估...
质量报告已保存到: quality_issues_report.csv

========== 数据质量评估结果 ==========
总记录数: 31889
发现问题记录数: 3 (占比 0.01%)

各类问题统计:
  年龄诊断矛盾: 3 条

【结论】✅ 数据质量良好，适合构建知识图谱（问题记录<10%）
正在处理临床数据...


2026-03-01 16:35:14,693 - __main__ - INFO - 成功读取临床Excel文件: d:\Users\LENOVO\Desktop\知识图谱\【脱敏数据】【2017-2024】门诊数据查询.xlsx，包含 7 个工作表
2026-03-01 16:35:14,693 - __main__ - INFO - 处理工作表: 2017，匹配列: {'tcm_diagnosis': '中医病名', 'tcm_syndrome': '中医证型', 'western_diagnosis': '诊断', 'symptom': '主诉', 'tongue_sign': '舌', 'pulse_sign': '脉', 'physical_sign': '体格检查', 'combined_prescription': '处方药物', 'chief_complaint': '主诉', 'present_illness': '现病史', 'past_history': '既往史'}


工作表 '2017': 已处理 100/3946 行
工作表 '2017': 已处理 200/3946 行
工作表 '2017': 已处理 300/3946 行
工作表 '2017': 已处理 400/3946 行
工作表 '2017': 已处理 500/3946 行
工作表 '2017': 已处理 600/3946 行
工作表 '2017': 已处理 700/3946 行
工作表 '2017': 已处理 800/3946 行
工作表 '2017': 已处理 900/3946 行
工作表 '2017': 已处理 1000/3946 行
工作表 '2017': 已处理 1100/3946 行
工作表 '2017': 已处理 1200/3946 行
工作表 '2017': 已处理 1300/3946 行
工作表 '2017': 已处理 1400/3946 行
工作表 '2017': 已处理 1500/3946 行
工作表 '2017': 已处理 1600/3946 行
工作表 '2017': 已处理 1700/3946 行
工作表 '2017': 已处理 1800/3946 行
工作表 '2017': 已处理 1900/3946 行
工作表 '2017': 已处理 2000/3946 行
工作表 '2017': 已处理 2100/3946 行
工作表 '2017': 已处理 2200/3946 行
工作表 '2017': 已处理 2300/3946 行
工作表 '2017': 已处理 2400/3946 行
工作表 '2017': 已处理 2500/3946 行
工作表 '2017': 已处理 2600/3946 行
工作表 '2017': 已处理 2700/3946 行
工作表 '2017': 已处理 2800/3946 行
工作表 '2017': 已处理 2900/3946 行
工作表 '2017': 已处理 3000/3946 行
工作表 '2017': 已处理 3100/3946 行
工作表 '2017': 已处理 3200/3946 行
工作表 '2017': 已处理 3300/3946 行
工作表 '2017': 已处理 3400/3946 行
工作表 '2017': 已处理 3500/3946 行
工作表 '2017': 已处理 3600/3946 行
工

2026-03-01 16:35:18,372 - __main__ - INFO - 处理工作表: 2018，匹配列: {'tcm_diagnosis': '中医病名', 'tcm_syndrome': '中医证型', 'western_diagnosis': '诊断', 'symptom': '主诉', 'tongue_sign': '舌', 'pulse_sign': '脉', 'physical_sign': '体格检查', 'combined_prescription': '处方药物', 'chief_complaint': '主诉', 'present_illness': '现病史', 'past_history': '既往史'}


工作表 '2017': 已处理 3800/3946 行
工作表 '2017': 已处理 3900/3946 行
工作表 '2018': 已处理 100/3985 行
工作表 '2018': 已处理 200/3985 行
工作表 '2018': 已处理 300/3985 行
工作表 '2018': 已处理 400/3985 行
工作表 '2018': 已处理 500/3985 行
工作表 '2018': 已处理 600/3985 行
工作表 '2018': 已处理 700/3985 行
工作表 '2018': 已处理 800/3985 行
工作表 '2018': 已处理 900/3985 行
工作表 '2018': 已处理 1000/3985 行
工作表 '2018': 已处理 1100/3985 行
工作表 '2018': 已处理 1200/3985 行
工作表 '2018': 已处理 1300/3985 行
工作表 '2018': 已处理 1400/3985 行
工作表 '2018': 已处理 1500/3985 行
工作表 '2018': 已处理 1600/3985 行
工作表 '2018': 已处理 1700/3985 行
工作表 '2018': 已处理 1800/3985 行
工作表 '2018': 已处理 1900/3985 行
工作表 '2018': 已处理 2000/3985 行
工作表 '2018': 已处理 2100/3985 行
工作表 '2018': 已处理 2200/3985 行
工作表 '2018': 已处理 2300/3985 行
工作表 '2018': 已处理 2400/3985 行
工作表 '2018': 已处理 2500/3985 行
工作表 '2018': 已处理 2600/3985 行
工作表 '2018': 已处理 2700/3985 行
工作表 '2018': 已处理 2800/3985 行
工作表 '2018': 已处理 2900/3985 行
工作表 '2018': 已处理 3000/3985 行
工作表 '2018': 已处理 3100/3985 行
工作表 '2018': 已处理 3200/3985 行
工作表 '2018': 已处理 3300/3985 行
工作表 '2018': 已处理 3400/3985 行
工

2026-03-01 16:35:21,952 - __main__ - INFO - 处理工作表: 2019，匹配列: {'tcm_diagnosis': '中医病名', 'tcm_syndrome': '中医证型', 'western_diagnosis': '诊断', 'symptom': '主诉', 'tongue_sign': '舌', 'pulse_sign': '脉', 'physical_sign': '体格检查', 'combined_prescription': '处方药物', 'chief_complaint': '主诉', 'present_illness': '现病史', 'past_history': '既往史'}


工作表 '2019': 已处理 100/3960 行
工作表 '2019': 已处理 200/3960 行
工作表 '2019': 已处理 300/3960 行
工作表 '2019': 已处理 400/3960 行
工作表 '2019': 已处理 500/3960 行
工作表 '2019': 已处理 600/3960 行
工作表 '2019': 已处理 700/3960 行
工作表 '2019': 已处理 800/3960 行
工作表 '2019': 已处理 900/3960 行
工作表 '2019': 已处理 1000/3960 行
工作表 '2019': 已处理 1100/3960 行
工作表 '2019': 已处理 1200/3960 行
工作表 '2019': 已处理 1300/3960 行
工作表 '2019': 已处理 1400/3960 行
工作表 '2019': 已处理 1500/3960 行
工作表 '2019': 已处理 1600/3960 行
工作表 '2019': 已处理 1700/3960 行
工作表 '2019': 已处理 1800/3960 行
工作表 '2019': 已处理 1900/3960 行
工作表 '2019': 已处理 2000/3960 行
工作表 '2019': 已处理 2100/3960 行
工作表 '2019': 已处理 2200/3960 行
工作表 '2019': 已处理 2300/3960 行
